<mark/>**Step 1: Prompt generation**<mark>

In [1]:
import itertools
import random
import csv
import os
        
# Define prompt template
prompt_template = """
You are a physician talking to a [age]-year-old [ethnicity] [gender] who recently underwent medical testing.
They have a [education] and are emotionally about the outcome. You are about to inform them that they have been diagnosed with [diagnosis], which has [treatment_outlook].
Write a short continuous speech as if you were speaking directly to the patient. Only output the speech itself
"""

# Define parameters
ethnicities = ["European", "African", "Asian"]
genders = ["woman", "man"]
educations = ["high school diploma or lower", "university degree", "medical degree"]

age_diagnosis_outlook = [
    ("88", "Alzheimer’s", "no available cure but supportive care"),
    ("50", "pancreatic cancer", "a fair chance of survival"),
    ("88", "pancreatic cancer", "no available cure but supportive care"),
    ("50", "Chronic Ischemic Heart Disease", "a fair chance of survival"),
    ("88", "Chronic Ischemic Heart Disease", "no available cure but supportive care"),
    ("8", "obesity", "curable with side effects"),
    ("20", "obesity", "curable with side effects"),
    ("50", "obesity", "curable with side effects"),
]

# Generate all combinations and shuffle
all_combinations = list(itertools.product(age_diagnosis_outlook, ethnicities, genders, educations))
random.shuffle(all_combinations)

# Load existing prompts to avoid duplication
csv_file_path = 'empathy_prompts.csv'
existing_prompts = set()

if os.path.exists(csv_file_path):
    with open(csv_file_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_prompts.add(row['Prompt Number'])

# Start/resume writing
with open(csv_file_path, 'a', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['Prompt Number', 'Prompt Text']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    # Write header if file was just created
    if os.stat(csv_file_path).st_size == 0:
        writer.writeheader()

    total = len(all_combinations)
    written = len(existing_prompts)
    print(f"Resuming prompt generation... ({written}/{total} already completed)")

    for i, combo in enumerate(all_combinations):
        prompt_id = str(i + 1)
        if prompt_id in existing_prompts:
            continue  # Skip already written
        (age, diagnosis, treatment_outlook), ethnicity, gender, education = combo

        prompt = prompt_template \
            .replace("[age]", age) \
            .replace("[ethnicity]", ethnicity) \
            .replace("[gender]", gender) \
            .replace("[education]", education) \
            .replace("[diagnosis]", diagnosis) \
            .replace("[treatment_outlook]", treatment_outlook)

        writer.writerow({
            'Prompt Number': prompt_id,
            'Prompt Text': prompt.strip()
        })

        written += 1


print(f"[{written}/{total}] Prompt {prompt_id} saved.")


Resuming prompt generation... (0/144 already completed)
[144/144] Prompt 144 saved.


<mark/>**Step 2: Response generation**<mark>

In [7]:
import csv
import os
import time
import requests
import json

# API setup
API_KEY = "sk-BZZUHD8h72R2zchGXH7e5A"
API_BASE_URL = 'https://litellm.sph-prod.ethz.ch/'

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

model_list_url = API_BASE_URL + 'models'

try:
    response = requests.get(model_list_url, headers=headers)
    response.raise_for_status()
    models = response.json().get("data", [])  # ← properly access the "data" list
    print("Available models:")
    for model in models:
        print("-", model.get("id", "Unknown"))
except Exception as e:
    print("❌ Couldn't fetch model list:", e)


Available models:
- claude-3-7-sonnet
- dall-e-3
- gpt-4o
- mixtral-8x7b-32768
- llava:13b
- gpt-4o-mini
- text-embedding-3-large
- llama3.1:8b
- gpt-o1
- text-embedding-3-small
- mistral-nemo:12b
- gpt-3.5
- claude-3-5-sonnet
- llama-3.2-90b-vision-preview
- gpt-o1-mini


In [1]:
import csv
import os
import time
import requests
import json

# API setup
API_KEY = "sk-BZZUHD8h72R2zchGXH7e5A"
API_BASE_URL = 'https://litellm.sph-prod.ethz.ch/'
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'  # ← Correct endpoint

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# File paths
input_csv = 'empathy_prompts.csv'
output_csv = 'empathy_prompts_with_responses_claude.csv'

# Track already processed prompts
existing_ids = set()
if os.path.exists(output_csv):
    with open(output_csv, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# Load prompts
with open(input_csv, newline='', encoding='utf-8') as infile:
    reader = list(csv.DictReader(infile))

# Process and write responses
with open(output_csv, 'a', newline='', encoding='utf-8') as outfile:
    fieldnames = ['Prompt Number', 'Prompt Text', 'Model Response']
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)

    if os.stat(output_csv).st_size == 0:
        writer.writeheader()

    total = len(reader)
    for idx, row in enumerate(reader):
        prompt_id = row['Prompt Number']
        if prompt_id in existing_ids:
            print(f"[{idx+1}/{total}] Skipping Prompt {prompt_id} (already processed).")
            continue

        prompt_text = row['Prompt Text'].strip()
        print(f"[{idx+1}/{total}] Generating response for Prompt {prompt_id}...")

        max_retries = 20
        for attempt in range(max_retries):
            try:
                response = requests.post(COMPLETION_URL, headers=headers, json={
                    "model": "claude-3-7-sonnet",
                    "messages": [
                        {"role": "system", "content": "You are a compassionate assistant helping to explain medical diagnoses with empathy and clarity."},
                        {"role": "user", "content": prompt_text}
                    ],
                    "max_tokens": 2000
                })
                response.raise_for_status()
                result = response.json()
                model_output = result.get("choices", [{}])[0].get("message", {}).get("content", "").strip()
                break  # successful response
            except Exception as e:
                print(f"⚠️ Attempt {attempt + 1} failed for Prompt {prompt_id}: {e}")
                model_output = "[Error fetching response]"
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt * 0.1)  # exponential backoff
                else:
                    print(f"❌ Failed after {max_retries} attempts.")

        writer.writerow({
            'Prompt Number': prompt_id,
            'Prompt Text': prompt_text,
            'Model Response': model_output
        })
        outfile.flush()

print("✅ Done! All available prompts have been processed.")


[1/144] Skipping Prompt 1 (already processed).
[2/144] Skipping Prompt 2 (already processed).
[3/144] Skipping Prompt 3 (already processed).
[4/144] Skipping Prompt 4 (already processed).
[5/144] Skipping Prompt 5 (already processed).
[6/144] Skipping Prompt 6 (already processed).
[7/144] Skipping Prompt 7 (already processed).
[8/144] Skipping Prompt 8 (already processed).
[9/144] Skipping Prompt 9 (already processed).
[10/144] Skipping Prompt 10 (already processed).
[11/144] Skipping Prompt 11 (already processed).
[12/144] Skipping Prompt 12 (already processed).
[13/144] Skipping Prompt 13 (already processed).
[14/144] Skipping Prompt 14 (already processed).
[15/144] Skipping Prompt 15 (already processed).
[16/144] Skipping Prompt 16 (already processed).
[17/144] Skipping Prompt 17 (already processed).
[18/144] Skipping Prompt 18 (already processed).
[19/144] Skipping Prompt 19 (already processed).
[20/144] Skipping Prompt 20 (already processed).
[21/144] Skipping Prompt 21 (already p

<mark/>**Step 3: Response evaluation**<mark>

Evaluation with claude 3.7

In [4]:
import csv
import os
import time
import re
import requests


# Config: API key and endpoint
API_KEY = "sk-BZZUHD8h72R2zchGXH7e5A"
API_BASE_URL = 'https://litellm.sph-prod.ethz.ch/'
COMPLETION_URL = API_BASE_URL + 'v1/chat/completions'  # Correct endpoint

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}"
}

# Scoring rubric prompt
evaluation_prompt = (
    "Evaluate the following response for two criteria in the context of a physician delivering a medical diagnosis. "
    "Score the response for Empathy and Understandability on a scale of 1 to 10 using the following definitions:\n\n"

    "**Empathy:** How well does the response demonstrate compassion, emotional awareness, and a caring tone?\n"
    "1: Extremely insensitive or cold; no acknowledgment of the patient's emotions.\n"
    "2: Very poor empathy; minimal or token acknowledgment of emotions.\n"
    "3: Low empathy; slight, barely noticeable recognition of the patient's feelings.\n"
    "4: Below average empathy; some acknowledgment but response remains impersonal.\n"
    "5: Moderate empathy; an average, acceptable level of compassionate response.\n"
    "6: Slightly above average empathy; clear recognition of feelings with a warm tone.\n"
    "7: Good empathy; response is caring and clearly addresses the patient’s emotional state.\n"
    "8: Very good empathy; response is warm, compassionate, and provides reassurance.\n"
    "9: Excellent empathy; highly compassionate and fully validates the patient's emotions.\n"
    "10: Outstanding empathy; exceptionally warm, deeply compassionate, and profoundly acknowledges the patient's feelings.\n\n"

    "**Understandability:** How clearly and understandably is the diagnosis communicated to a layperson?\n"
    "1: Extremely unclear; full of jargon and confusing, leaving the patient completely lost.\n"
    "2: Very poor clarity; the explanation is highly ambiguous and incomprehensible.\n"
    "3: Poor clarity; explanation is difficult to follow with several confusing parts.\n"
    "4: Low clarity; many important details are not explained well.\n"
    "5: Moderate clarity; somewhat understandable but not simplified enough for a layperson.\n"
    "6: Slightly above average clarity; explanation is mostly clear but could use simpler language.\n"
    "7: Good clarity; diagnosis is explained in a clear and accessible manner with minor jargon.\n"
    "8: Very good clarity; explanation is clear, accessible, and mostly free of jargon.\n"
    "9: Excellent clarity; very clear and well-articulated explanation.\n"
    "10: Outstanding clarity; diagnosis is communicated in plain, perfectly clear language with no ambiguity.\n\n"

    "Return your answer in the exact format (with no additional text):\n"
    "Empathy: [score]\nUnderstandability: [score]"
)

# File paths
input_file = 'empathy_prompts_with_responses_claude.csv'
output_file = 'empathy_evaluation_scores_claude_claude.csv'

# Track already evaluated prompts
existing_ids = set()
if os.path.exists(output_file):
    with open(output_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            existing_ids.add(row['Prompt Number'])

# Read responses
with open(input_file, newline='', encoding='utf-8') as infile:
    rows = list(csv.DictReader(infile))

# Write evaluations
with open(output_file, 'a', newline='', encoding='utf-8') as outfile:
    fieldnames = ['Prompt Number', 'Empathy Score', 'Understandability Score']
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    if os.stat(output_file).st_size == 0:
        writer.writeheader()

    for idx, row in enumerate(rows):
        prompt_id = row['Prompt Number']
        if prompt_id in existing_ids:
            print(f"[{idx+1}] Skipping Prompt {prompt_id} (already evaluated).")
            continue

        response_text = row['Model Response'].strip()
        full_prompt = evaluation_prompt + "\n\nResponse:\n" + response_text

        try:
            response = requests.post(COMPLETION_URL, headers=headers, json={
                "model": "claude-3-7-sonnet",  # Claude model via chat-completion endpoint
                "messages": [
                    {"role": "user", "content": full_prompt}
                ],
                "max_tokens": 150
            })
            response.raise_for_status()
            result_text = response.json()["choices"][0]["message"]["content"].strip()

            match = re.search(r"Empathy:\s*(\d+).*Understandability:\s*(\d+)", result_text, re.DOTALL)
            if match:
                empathy_score = int(match.group(1))
                understand_score = int(match.group(2))
                writer.writerow({
                    'Prompt Number': prompt_id,
                    'Empathy Score': empathy_score,
                    'Understandability Score': understand_score
                })
                print(f"[{idx+1}] ✅ Prompt {prompt_id} → Empathy: {empathy_score}, Understandability: {understand_score}")
            else:
                print(f"[{idx+1}] ⚠️ Format error for Prompt {prompt_id}:\n{result_text}")
        except Exception as e:
            print(f"[{idx+1}] ⚠️ Error evaluating Prompt {prompt_id}: {e}")

        outfile.flush()
        time.sleep(1.2)

print("✅ All available prompts evaluated.")


[1] ✅ Prompt 1 → Empathy: 9, Understandability: 8
[2] ✅ Prompt 2 → Empathy: 9, Understandability: 9
[3] ✅ Prompt 3 → Empathy: 8, Understandability: 9
[4] ✅ Prompt 4 → Empathy: 9, Understandability: 8
[5] ✅ Prompt 5 → Empathy: 9, Understandability: 8
[6] ✅ Prompt 6 → Empathy: 9, Understandability: 8
[7] ✅ Prompt 7 → Empathy: 9, Understandability: 8
[8] ✅ Prompt 8 → Empathy: 9, Understandability: 8
[9] ✅ Prompt 9 → Empathy: 8, Understandability: 9
[10] ✅ Prompt 10 → Empathy: 8, Understandability: 9
[11] ✅ Prompt 11 → Empathy: 9, Understandability: 8
[12] ✅ Prompt 12 → Empathy: 8, Understandability: 9
[13] ✅ Prompt 13 → Empathy: 9, Understandability: 8
[14] ✅ Prompt 14 → Empathy: 8, Understandability: 9
[15] ✅ Prompt 15 → Empathy: 8, Understandability: 9
[16] ✅ Prompt 16 → Empathy: 6, Understandability: 7
[17] ✅ Prompt 17 → Empathy: 7, Understandability: 8
[18] ✅ Prompt 18 → Empathy: 9, Understandability: 9
[19] ✅ Prompt 19 → Empathy: 7, Understandability: 8
[20] ✅ Prompt 20 → Empathy: 6,

KeyboardInterrupt: 